# 03 — Explanations from the fine-tuned Llama

Asks the **already fine-tuned** Llama 3.1 8B why it made its decisions. **Nothing is trained here**: it loads the adapters that `02_finetune.ipynb` saved to Drive, and writes only to `results/explanations/`, so no earlier result can be overwritten. The logic lives in `src/explain.py`.

- **Part A — hide-and-see:** hide the sender, subject or body (or the word "Enron") and measure how the decision changes.
- **Part B — written explanations:** three ways of getting text (`prefill`, `followup`, and the control `base_prefill`), explained at the top of `src/explain.py`.

Zero-shot Llama is not run again for its own explanations: they were saved by `01_baselines.ipynb`.

**Run it twice** (rough times on an L4):

| `TRIAL` | What runs | Time |
|---|---|---|
| `True` (default) | 20 emails with the main model | ~10-15 min, mostly setup |
| `False` | 150 emails with the main model + 49 Nazario misses with the no-Nazario model | ~30-45 min |

### Before you run
1. **GPU runtime:** *Runtime → Change runtime type → L4* (or A100). A T4 works but is ~3x slower; set `BATCH_SIZE = 4`.
2. **Hugging Face token and Llama license:** same as before (`HF_TOKEN` secret).
3. *Runtime → Run all.* If Colab disconnects, run all again: finished emails are skipped.
4. After `CUDA out of memory` or *"GPU memory is still in use"*: *Runtime → **Restart session***, then *Run all*.

## 1. Setup — check the GPU, get the code, install libraries

In [ ]:
# Show which GPU Colab gave us. If this fails, the runtime has no GPU —
# change it under Runtime → Change runtime type.
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os, sys

# Let PyTorch grow its GPU memory pool in flexible chunks, which reduces
# out-of-memory errors. Must be set BEFORE torch is imported.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Get our project code from GitHub (or update it if already there).
REPO_DIR = "/content/llm-phishing"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/mihaighise/llm-phishing.git {REPO_DIR}

# Install / upgrade the libraries:
#   transformers  — loads the model and its chat template
#   accelerate    — places the model on the GPU
#   bitsandbytes  — 4-bit quantization (the "Q" in QLoRA)
#   peft          — LoRA adapters (the "LoRA" in QLoRA)
!pip install -q -U transformers accelerate bitsandbytes peft

# Make `import explain` / `import inference` work.
sys.path.insert(0, f"{REPO_DIR}/src")

## 2. Log in to Hugging Face
Needed to download the gated Llama model. Reads the `HF_TOKEN` secret.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# userdata.get reads a Colab secret, so the token never appears in the notebook.
login(token=userdata.get("HF_TOKEN"))

## 3. Build the dataset on Colab
Same as in the other notebooks. Only the test set is used here, so only its fingerprint is checked. If it differs from the Mac build, the notebook stops, because the saved decisions would belong to different emails.

In [ ]:
import hashlib
from pathlib import Path

RAW_DIR = Path(REPO_DIR) / "data" / "zenodo"
DATA_DIR = Path(REPO_DIR) / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download each source CSV unless it is already present.
for name in ["Nazario", "Nigerian_Fraud", "Nazario_5", "Nigerian_5", "SpamAssasin"]:
    target = RAW_DIR / f"{name}.csv"
    if not target.exists():
        url = f"https://zenodo.org/api/records/8339691/files/{name}.csv/content"
        !curl -sfL -o {target} "{url}"

# Build the splits (same command you run locally).
!python3 {REPO_DIR}/src/build_dataset.py

# Fingerprint (MD5 hash) of the test set built on the Mac.
EXPECTED = "50b428d99f00fc8bf320d827acd4aefc"
actual = hashlib.md5((DATA_DIR / "test.jsonl").read_bytes()).hexdigest()
if actual != EXPECTED:
    raise RuntimeError("test.jsonl differs from the local build!")
print("test.jsonl: OK — identical to local build")

## 4. Google Drive: find the saved adapters
Stops with a clear message if an adapter is missing. It never trains one.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/llm-phishing")
# A NEW folder, so nothing here can be mixed up with results/raw/.
OUT_DIR = DRIVE_DIR / "results" / "explanations"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_MODEL = "llama31_8b_ft_train_3000"
NO_NAZARIO_MODEL = "llama31_8b_ft_train_1500_no_nazario"

def adapter_dir(run_name):
    """Where 02_finetune.ipynb saved this model's adapter."""
    return DRIVE_DIR / "finetune" / run_name / "adapter"

for run_name in (MAIN_MODEL, NO_NAZARIO_MODEL):
    for fname in ("adapter_config.json", "adapter_model.safetensors"):
        if not (adapter_dir(run_name) / fname).exists():
            raise FileNotFoundError(f"Missing: {adapter_dir(run_name) / fname}")
    print(f"{run_name}: adapter found")

## 5. Configuration
Everything you might want to change is in this one cell.

In [ ]:
# Make sure Python can find our modules in src/, also after a runtime restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import json
from prompts import PROMPT_VERSION

TRIAL = True      # True = the 20-email trial; False = everything
BATCH_SIZE = 16   # emails at once; 16 on an L4/A100, 4 on a T4

# Which emails to explain. Made on the Mac by src/explain_select.py.
selection = json.loads(
    Path("/content/llm-phishing/results/explanation_emails.json").read_text())
# The saved decisions must come from the prompt this code still uses.
assert selection["prompt_version"] == PROMPT_VERSION, "prompt version changed"

with open(DATA_DIR / "test.jsonl", encoding="utf-8") as f:
    test_emails = {e["id"]: e for e in map(json.loads, f)}

def emails_for(run_name):
    """The emails to run for one model, and what we know about each of them."""
    entries = {e["id"]: e for e in selection["models"][run_name]
               if e["trial"] or not TRIAL}
    return [test_emails[i] for i in entries], entries

def out_path(run_name, kind):
    """e.g. .../llama31_8b_ft_train_3000_scores_v2.jsonl"""
    return OUT_DIR / f"{run_name}_{kind}_{PROMPT_VERSION}.jsonl"

for run_name in (MAIN_MODEL, NO_NAZARIO_MODEL):
    print(run_name, "->", len(emails_for(run_name)[0]), "emails")

## 6. Run the main fine-tuned model (`train_3000`)
Part A (scores), then Part B (text).

In [ ]:
# Make sure Python can find our modules in src/, also after a runtime restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from inference import load_model, free_gpu_memory, check_gpu_memory
from explain import score_emails, explain_emails

def run_all(run_name):
    """Load one fine-tuned model, do Part A and Part B, free the GPU again."""
    emails, entries = emails_for(run_name)
    if not emails:  # the no-Nazario model has no trial emails
        print(f"{run_name}: nothing to run")
        return
    # Loading the 4-bit model needs ~6 GB; stop clearly if the GPU is taken.
    check_gpu_memory(min_free_gb=8)
    model, tokenizer = load_model("llama31_8b",
                                  adapter_path=str(adapter_dir(run_name)))
    score_emails(run_name, model, tokenizer, emails, entries,
                 out_path(run_name, "scores"), batch_size=BATCH_SIZE)
    explain_emails(run_name, model, tokenizer, emails, entries,
                   out_path(run_name, "explanations"), batch_size=BATCH_SIZE)
    # `model` only exists inside this function, so after `del` nothing refers
    # to it any more and its GPU memory can be released.
    del model, tokenizer
    free_gpu_memory()

run_all(MAIN_MODEL)

## 7. Run the model trained without Nazario
Only in the full run: explains the 49 Nazario phishing emails that this model missed.

In [ ]:
run_all(NO_NAZARIO_MODEL)

## 8. Look at the results
- **Agreement:** the fine-tuned score on the full email should match the official answer for (nearly) every email. If not, something is wrong with the setup.
- **Usable:** the share of answers that contain real text.

In [ ]:
import pandas as pd
from explain import load_rows

scores, texts = [], []
for run_name in (MAIN_MODEL, NO_NAZARIO_MODEL):
    scores += load_rows(out_path(run_name, "scores"))
    texts += load_rows(out_path(run_name, "explanations"))
scores, texts = pd.DataFrame(scores), pd.DataFrame(texts)

# --- Sanity check: does the score agree with the official answer?
full = scores[(scores.scorer == "finetuned") & (scores.variant == "full")]
print(f"Agreement with the saved answers: "
      f"{(full.label == full.saved_label).sum()}/{len(full)}")

# --- Part A: how often does hiding a part flip the model's decision?
# We compare each variant with the SAME model's decision on the full email.
keys = ["model", "scorer", "id"]
base = scores[scores.variant == "full"][keys + ["label", "log_odds"]]
merged = scores[scores.variant != "full"].merge(
    base, on=keys, suffixes=("", "_full"))
merged["flipped"] = merged.label != merged.label_full
merged["change"] = merged.log_odds - merged.log_odds_full  # + = more phishing
display(merged.groupby(["model", "group", "scorer", "variant"]).agg(
    emails=("id", "size"), flipped=("flipped", "sum"),
    mean_change=("change", "mean")).round(2))

# --- Part B: did we get real text?
display(texts.groupby(["model", "method", "group"]).agg(
    emails=("id", "size"), usable=("usable", "sum"),
    finished=("finished", "sum")))

In [ ]:
# Read some explanations side by side: the first 20 emails of the main model.
main = texts[texts.model == MAIN_MODEL]
for email_id in list(dict.fromkeys(main.id))[:20]:
    email = test_emails[email_id]
    print("=" * 90)
    print(f"{email_id[:8]}  true: {email['label']}  ({email['source']})")
    print(f"From: {email['sender']}\nSubject: {email['subject']}")
    print(email["body"][:400], "\n")
    for row in main[main.id == email_id].itertuples():
        print(f"[{row.method}] {row.label}: {row.reasoning}\n")

## 9. What to do next
1. **After the trial:** send the output of section 8 to Claude, to decide whether the text is good enough.
2. **Full run:** set `TRIAL = False` in section 5 and *Run all* again. The 20 trial emails are kept and skipped.
3. Copy the files from `MyDrive/llm-phishing/results/explanations/` to `results/explanations/` on the Mac.